In [15]:
import os
import re
import warnings
import random
import sys
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    roc_auc_score, accuracy_score, average_precision_score,
    precision_score, recall_score, f1_score,
    confusion_matrix, brier_score_loss, roc_curve
)

from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from scipy.stats import ks_2samp
import shap
import joblib

# Suppress warnings
warnings.filterwarnings("ignore", category=UserWarning, module="sklearn.preprocessing._encoders")
warnings.filterwarnings("ignore")

# CONFIG
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

BASE_DIR = Path("/Users/amanda/Desktop/UCBT")
INPUT_PATH = BASE_DIR / "ucbt_dataset_synthetic_best.csv"
REAL_PATH = BASE_DIR / "ucbt_dataset.csv" 
FIG_DIR = BASE_DIR / "figs_auc"
FIG_DIR.mkdir(exist_ok=True, parents=True)
OUT_DIR = BASE_DIR / "models_output" / "metrics"
OUT_DIR.mkdir(exist_ok=True, parents=True)

# Dose filters and ALL-only for Survival
#H1
#USE_DOSE_FILTERS = True
#CD34_MIN, TNC_MIN = 2.5, 3.5
#CD34_MAX_PLATELET, TNC_MAX_PLATELET = 6.5, 8.6 
#CD34_MAX_SURVIVAL, TNC_MAX_SURVIVAL = 6.8, 9.5
#USE_ALL_ONLY_SURVIVAL = True

#H2
#USE_DOSE_FILTERS = True
#CD34_MIN, TNC_MIN = 2.5, 3.5
#CD34_MAX_PLATELET, TNC_MAX_PLATELET = 6.5, 8.6 
#CD34_MAX_SURVIVAL, TNC_MAX_SURVIVAL = 6.8, 9.5
#USE_ALL_ONLY_SURVIVAL = False

#H3
#USE_DOSE_FILTERS = False
#CD34_MIN, TNC_MIN = 2.5, 3.5
#CD34_MAX_PLATELET, TNC_MAX_PLATELET = 6.5, 8.6 
#CD34_MAX_SURVIVAL, TNC_MAX_SURVIVAL = 6.8, 9.5
#USE_ALL_ONLY_SURVIVAL = True

#H4
#USE_DOSE_FILTERS = False
#CD34_MIN, TNC_MIN = 2.5, 3.5
#CD34_MAX_PLATELET, TNC_MAX_PLATELET = 6.5, 8.6 
#CD34_MAX_SURVIVAL, TNC_MAX_SURVIVAL = 6.8, 9.5
#USE_ALL_ONLY_SURVIVAL = False

#H5
#USE_DOSE_FILTERS = True
#CD34_MIN, TNC_MIN = 3.0, 4.0
#CD34_MAX_PLATELET, TNC_MAX_PLATELET = 6.5, 8.6 
#CD34_MAX_SURVIVAL, TNC_MAX_SURVIVAL = 6.8, 9.5
#USE_ALL_ONLY_SURVIVAL = True

#H6
USE_DOSE_FILTERS = True
CD34_MIN, TNC_MIN = 3.0, 4.0
CD34_MAX_PLATELET, TNC_MAX_PLATELET = 6.5, 8.6 
CD34_MAX_SURVIVAL, TNC_MAX_SURVIVAL = 6.8, 9.5
USE_ALL_ONLY_SURVIVAL = False

#H7
#USE_DOSE_FILTERS = True
#CD34_MIN, TNC_MIN = 2.0, 3.0
#CD34_MAX_PLATELET, TNC_MAX_PLATELET = 7.5, 9.6
#CD34_MAX_SURVIVAL, TNC_MAX_SURVIVAL = 7.8, 10.5
#USE_ALL_ONLY_SURVIVAL = True

# Bagging
N_BAGS = 5
# Cross-validation
N_FOLDS = 10 # Increased for stability
MIN_SAMPLES_CV = 10
# Minimum group size for plots
MIN_GROUP_N = 20
# SHAP top k features
SHAP_TOP_K = 6
# Holdout set proportion
HOLDOUT_SIZE = 0.2

# Unique naming helpers
RUN_DATE = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")
DATASET_TAG = "synthetic" if "synthetic" in INPUT_PATH.name.lower() else "real"

def out_csv(stem: str) -> Path:
    """CSV in OUT_DIR: <stem>_<dataset>_<yyyymmdd_hhmmss>.csv"""
    return OUT_DIR / f"{stem}_{DATASET_TAG}_{RUN_DATE}.csv"

def out_png(stem: str) -> Path:
    """PNG in OUT_DIR: <stem>_<dataset>_<yyyymmdd_hhmmss>.png"""
    return OUT_DIR / f"{stem}_{DATASET_TAG}_{RUN_DATE}.png"

def out_pkl(stem: str) -> Path:
    """PKL in OUT_DIR: <stem>_<dataset>_<yyyymmdd_hhmmss>.pkl"""
    return OUT_DIR / f"{stem}_{DATASET_TAG}_{RUN_DATE}.pkl"

def out_json(stem: str) -> Path:
    """JSON in OUT_DIR: <stem>_<dataset>_<yyyymmdd_hhmmss>.json"""
    return OUT_DIR / f"{stem}_{DATASET_TAG}_{RUN_DATE}.json"


# ENVIRONMENT CHECK
print(f"Python executable: {sys.executable}")
try:
    print(f"SHAP version: {shap.__version__}")
except AttributeError:
    print("SHAP not properly installed. Please reinstall: pip install shap==0.46.0")
    

# SYNTHETIC DATA VALIDATION
def validate_synthetic_data(df_synthetic, df_real):
    print("\nValidating synthetic data with Kolmogorov-Smirnov tests...")
    ks_results = []
    # Numerical columns
    num_cols = ["CD34_num", "TNC_num", "Recipient_Age"]
    for col in num_cols:
        if col in df_synthetic.columns and col in df_real.columns:
            stat, pval = ks_2samp(df_synthetic[col].dropna(), df_real[col].dropna())
            ks_results.append({"column": col, "ks_stat": stat, "p_value": pval})
            print(f"KS test for {col}: stat={stat:.4f}, p={pval:.4f}")
    # Categorical columns
    cat_cols = ["HLA_Match_Level", "Disease_Type", "Conditioning_Regimen", "Ethnicity", "Race"]
    for col in cat_cols:
        if col in df_synthetic.columns and col in df_real.columns:
            synth_dist = df_synthetic[col].value_counts(normalize=True).to_dict()
            real_dist = df_real[col].value_counts(normalize=True).to_dict()
            categories = set(synth_dist.keys()) | set(real_dist.keys())
            synth_vec = [synth_dist.get(cat, 0) for cat in categories]
            real_vec = [real_dist.get(cat, 0) for cat in categories]
            stat, pval = ks_2samp(synth_vec, real_vec)
            ks_results.append({"column": col, "ks_stat": stat, "p_value": pval})
            print(f"KS test for {col} distribution: stat={stat:.4f}, p={pval:.4f}")
    ks_df = pd.DataFrame(ks_results)
    ks_path = out_csv("ks_validation_results")
    ks_df.to_csv(ks_path, index=False)
    print(f"[saved] {ks_path}")

    return ks_df

# UTILITIES
def make_safe_filename(s: str) -> str:
    s = re.sub(r"[^\w\-\.]+", "_", s)
    s = re.sub(r"_+", "_", s).strip("_")
    return s[:200] if len(s) > 200 else s
    
def prevalence(y: np.ndarray) -> float:
    y = np.asarray(y).astype(int)
    return float(np.mean(y))
    
def best_threshold_for_accuracy(y_true, proba, lo=0.05, hi=0.95, steps=901):
    grid = np.linspace(lo, hi, steps)
    acc = [accuracy_score(y_true, (proba >= t).astype(int)) for t in grid]
    i = int(np.argmax(acc))
    return float(grid[i]), float(acc[i])
    
def build_preprocessor(cat_cols, num_cols, categories):
    cat_tf = Pipeline([
        ("imp", SimpleImputer(strategy="most_frequent")),
        ("ohe", OneHotEncoder(categories=categories, handle_unknown="ignore", sparse_output=False))
    ])
    num_tf = Pipeline([
        ("imp", SimpleImputer(strategy="median")),
        ("sc", MinMaxScaler())
    ])
    pre = ColumnTransformer(
        transformers=[
            ("num", num_tf, num_cols),
            ("cat", cat_tf, cat_cols)
        ],
        remainder="drop"
    )
    return pre
    
def evaluate(y_true, proba, prefix=""):
    thr, acc = best_threshold_for_accuracy(y_true, proba)
    y_pred = (proba >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return {
        f'{prefix}acc': acc,
        f'{prefix}auc': roc_auc_score(y_true, proba),
        f'{prefix}pr_auc': average_precision_score(y_true, proba),
        f'{prefix}brier': brier_score_loss(y_true, proba),
        f'{prefix}thr': thr,
        f'{prefix}prec': precision_score(y_true, y_pred, zero_division=0),
        f'{prefix}rec': recall_score(y_true, y_pred, zero_division=0),
        f'{prefix}f1': f1_score(y_true, y_pred, zero_division=0),
        f'{prefix}tn': int(tn), f'{prefix}fp': int(fp), f'{prefix}fn': int(fn), f'{prefix}tp': int(tp)
    }
    
def shap_feature_selection(model, X_tr, colnames, top_k=SHAP_TOP_K):
    try:
        explainer = shap.TreeExplainer(model)
        shap_values = explainer.shap_values(X_tr)
        shap_sum = np.abs(shap_values).mean(axis=0)
        if len(shap_sum) != len(colnames):
            print(f"Warning: SHAP sum length ({len(shap_sum)}) does not match colnames length ({len(colnames)}), using all features")
            return colnames, list(range(len(colnames)))
        feature_importance = pd.DataFrame({'feature': colnames, 'importance': shap_sum})
        top_features = feature_importance.nlargest(top_k, 'importance')['feature'].tolist()
        top_indices = [colnames.index(f) for f in top_features if f in colnames]
        print(f"SHAP succeeded: Top features: {top_features}")
        return top_features, top_indices
    except Exception as e:
        print(f"SHAP failed: {e}, using all features")
        return colnames, list(range(len(colnames)))
        
def subgroup_block(df, y_true, proba, thr, col, task_name):
    out = []
    if col not in df.columns:
        return pd.DataFrame()
    cats = df[col].astype(str).fillna('NA').value_counts().index.tolist()
    for k in cats:
        idx = (df[col].astype(str).fillna('NA') == k)
        if idx.sum() < MIN_GROUP_N:
            continue
        yt, pt = y_true[idx], proba[idx]
        auc_s = roc_auc_score(yt, pt) if len(np.unique(yt)) > 1 else np.nan
        acc_s = accuracy_score(yt, (pt >= thr).astype(int))
        ap_s = average_precision_score(yt, pt) if len(np.unique(yt)) > 1 else np.nan
        out.append({'group': col, 'level': k, 'n': int(idx.sum()), 'auc': auc_s, 'acc': acc_s, 'ap': ap_s})
    return pd.DataFrame(out)
    
def plot_roc(model, X, y, title, path):
    prob = model.predict_proba(X)[:, 1]
    fpr, tpr, _ = roc_curve(y, prob)
    auc = roc_auc_score(y, prob)
    plt.figure(figsize=(5.5, 4.5))
    plt.plot(fpr, tpr, label=f"AUC={auc:.3f}")
    plt.plot([0, 1], [0, 1], "--")
    plt.xlabel("FPR"); plt.ylabel("TPR"); plt.title(title); plt.legend()
    plt.tight_layout(); plt.savefig(path, dpi=220, bbox_inches="tight"); plt.close()
    
def plot_acc_prev_for_group(df_plot, y_true, proba, group_col, thr, title_prefix, out_dir):
    if group_col not in df_plot.columns:
        return
    data = pd.DataFrame({
        "group": df_plot[group_col].astype(str),
        "y": y_true.astype(int),
        "pred": (proba >= thr).astype(int)
    })
    stats = (
        data.groupby("group")
        .agg(
            n=("y", "size"),
            prev=("y", "mean"),
            acc=("pred", lambda s: (s.values == data.loc[s.index, "y"].values).mean())
        )
        .reset_index()
    )
    stats = stats.sort_values("prev", ascending=False)
    stats = stats[stats["n"] >= MIN_GROUP_N]
    if stats.empty:
        return
    fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=False)
    plt.suptitle(f"{title_prefix} — {group_col}")
    axes[0].bar(stats["group"], stats["acc"])
    axes[0].set_title("Accuracy by Group")
    axes[0].set_xticklabels(stats["group"], rotation=45, ha="right")
    axes[0].set_ylim(0, 1)
    for i, (g, acc, n) in enumerate(zip(stats["group"], stats["acc"], stats["n"])):
        axes[0].text(i, acc + 0.02, f"n={n}", ha="center", va="bottom", fontsize=9)
    axes[1].bar(stats["group"], stats["prev"])
    axes[1].set_title("Prevalence by Group")
    axes[1].set_xticklabels(stats["group"], rotation=45, ha="right")
    axes[1].set_ylim(0, 1)
    for i, (g, prev, n) in enumerate(zip(stats["group"], stats["prev"], stats["n"])):
        axes[1].text(i, prev + 0.02, f"n={n}", ha="center", va="bottom", fontsize=9)
    for ax in axes:
        ax.grid(axis="y", alpha=0.2)
    fname = make_safe_filename(f"{title_prefix}_acc_prev_{group_col}.png")
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.savefig(out_dir / fname, dpi=220, bbox_inches="tight")
    plt.close()
    print(f"[saved] {out_dir / fname}")

# SAFE FIT
def safe_fit_xgb(X_tr, y_tr, params=None):
    params = dict(params or {})
    pos = int(np.sum(y_tr))
    spw = params.get('scale_pos_weight', (len(y_tr) - pos) / max(pos, 1) if pos > 0 else 1.0)
    p = {
        'objective': "binary:logistic",
        'eval_metric': "auc",
        'random_state': params.pop("random_state", RANDOM_STATE),
        'n_jobs': -1,
        'tree_method': "hist",
        'scale_pos_weight': spw
    }
    p.update(params)
    model = XGBClassifier(**p)
    model.fit(X_tr, y_tr, verbose=False)
    return model

# CROSS-VALIDATION (LEAK-FREE)
def cv_with_threshold(X_df, y, task_name, outcome_col, cat_cols, num_cols, categories, best_params):
    print(f"[{task_name}] Running {N_FOLDS}-fold CV (leak-free)...")
    cv_rows = []
    if len(X_df) < MIN_SAMPLES_CV or len(np.unique(y)) < 2:
        print(f"[{task_name}] CV skipped: insufficient samples or single class.")
        return {
            "auc_mean": np.nan, "auc_std": np.nan,
            "acc_mean": np.nan, "acc_std": np.nan,
            "prec_mean": np.nan, "prec_std": np.nan,
            "rec_mean": np.nan, "rec_std": np.nan,
            "best_spw": best_params.get('scale_pos_weight', 1.0)
        }

    cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    spw_scores, cv_metrics = [], []

    for fold, (tr, te) in enumerate(cv.split(X_df, y), 1):
        X_tr, X_te = X_df.iloc[tr], X_df.iloc[te]
        y_tr, y_te = y[tr], y[te].astype(int)

        # Fit preprocessing *inside* the fold
        pre = build_preprocessor(cat_cols, num_cols, categories)
        X_tr_mat = pre.fit_transform(X_tr)
        X_te_mat = pre.transform(X_te)
        colnames = list(pre.get_feature_names_out())

        # SMOTE only on training set
        sm = SMOTE(random_state=RANDOM_STATE)
        try:
            X_tr_mat, y_tr = sm.fit_resample(X_tr_mat, y_tr)
        except ValueError:
            pass  # skip if imbalance too small

        # SHAP feature selection on training only
        xgb_temp = safe_fit_xgb(X_tr_mat, y_tr, params=best_params)
        top_features, top_indices = shap_feature_selection(xgb_temp, X_tr_mat, colnames)
        X_tr_mat = X_tr_mat[:, top_indices]
        X_te_mat = X_te_mat[:, top_indices]

        # Tune scale_pos_weight within fold
        spw_grid = [0.5, 0.75, 1.0, 1.25, 1.5]
        best_spw, best_auc = None, 0.0
        for spw in spw_grid:
            temp_params = best_params.copy()
            temp_params['scale_pos_weight'] = spw
            model = safe_fit_xgb(X_tr_mat, y_tr, params=temp_params)
            probs = model.predict_proba(X_te_mat)[:, 1]
            auc = roc_auc_score(y_te, probs)
            if auc > best_auc:
                best_auc, best_spw = auc, spw
            spw_scores.append({'fold': fold, 'spw': spw, 'auc': auc})

        # Fit with best_spw
        model = safe_fit_xgb(X_tr_mat, y_tr, params={**best_params, 'scale_pos_weight': best_spw})
        proba_te = model.predict_proba(X_te_mat)[:, 1]
        metrics = evaluate(y_te, proba_te, prefix="cv_")
        cv_metrics.append(metrics)
        print(f"[{task_name}] Fold {fold}: AUC={metrics['cv_auc']:.3f}, ACC={metrics['cv_acc']:.3f}, best_spw={best_spw}")

    cv_df = pd.DataFrame(cv_metrics)
    cv_summary = {
        "auc_mean": cv_df["cv_auc"].mean(),
        "auc_std": cv_df["cv_auc"].std(),
        "acc_mean": cv_df["cv_acc"].mean(),
        "acc_std": cv_df["cv_acc"].std(),
        "prec_mean": cv_df["cv_prec"].mean(),
        "prec_std": cv_df["cv_prec"].std(),
        "rec_mean": cv_df["cv_rec"].mean(),
        "rec_std": cv_df["cv_rec"].std(),
    }

    spw_df = pd.DataFrame(spw_scores)
    if not spw_df.empty:
        cv_summary["best_spw"] = spw_df.groupby("spw")["auc"].mean().idxmax()
    else:
        cv_summary["best_spw"] = best_params.get("scale_pos_weight", 1.0)

    print(f"[{task_name}] CV Summary: AUC={cv_summary['auc_mean']:.3f}±{cv_summary['auc_std']:.3f}, "
          f"ACC={cv_summary['acc_mean']:.3f}±{cv_summary['acc_std']:.3f}")
    return cv_summary

# TASK RUNNER
def run_task(task_name, outcome_col, train_mask_fn, eval_mask_fn, cat_cols, num_cols, exclude_cols=None, categories=None):
    print(f"\n[{task_name}] Preparing data...")
    exclude_cols = exclude_cols or []
    full_mask = df[outcome_col].notna()
    X_df_full = df[full_mask].copy()
    if outcome_col not in df.columns:
        raise ValueError(f"Outcome '{outcome_col}' not found in data.")
    filtered_mask = train_mask_fn(df) & eval_mask_fn(df)
    X_df_filtered = df[filtered_mask].copy()
    X_df_full = X_df_full[X_df_full[outcome_col].notna() & X_df_full[cat_cols + num_cols].notna().all(axis=1)].reset_index(drop=True)
    y_full = X_df_full[outcome_col].astype(int).values
    print(f"[{task_name}] Full dataset size: {len(X_df_full)}")
    X_df_filtered = X_df_filtered[X_df_filtered[outcome_col].notna() & X_df_filtered[cat_cols + num_cols].notna().all(axis=1)].reset_index(drop=True)
    y_filtered = X_df_filtered[outcome_col].astype(int).values
    print(f"[{task_name}] Filtered size: {len(X_df_filtered)}")
    class_counts_full = pd.Series(y_full).value_counts()
    class_counts_filtered = pd.Series(y_filtered).value_counts()
    print(f"[{task_name}] Full dataset class counts: {class_counts_full.to_dict()}")
    print(f"[{task_name}] Filtered dataset class counts: {class_counts_filtered.to_dict()}")
    if len(class_counts_filtered) < 2:
        raise ValueError(f"[{task_name}] Only one class present in filtered target variable.")
    for col in cat_cols:
        if col in X_df_full.columns:
            print(f"[{task_name}] Unique {col} in full dataset: {X_df_full[col].astype(str).unique().tolist()}")
        if col in X_df_filtered.columns:
            print(f"[{task_name}] Unique {col} in filtered dataset: {X_df_filtered[col].astype(str).unique().tolist()}")
    if "CD34_num" in X_df_filtered.columns:
        cd34_vals = pd.to_numeric(X_df_filtered["CD34_num"], errors="coerce")
        print(f"[{task_name}] CD34_num in filtered dataset: min={cd34_vals.min():.2f}, max={cd34_vals.max():.2f}, nulls={cd34_vals.isna().sum()}")
    if "TNC_num" in X_df_filtered.columns:
        tnc_vals = pd.to_numeric(X_df_filtered["TNC_num"], errors="coerce")
        print(f"[{task_name}] TNC_num in filtered dataset: min={tnc_vals.min():.2f}, max={tnc_vals.max():.2f}, nulls={tnc_vals.isna().sum()}")
    used_cats = [c for c in cat_cols if c in X_df_full.columns and c not in exclude_cols]
    used_nums = [c for c in num_cols if c in X_df_full.columns and c not in exclude_cols]
    X_temp, X_te_filtered, y_temp, y_te_filtered = train_test_split(
        X_df_filtered[used_cats + used_nums], y_filtered, test_size=0.2, stratify=y_filtered, random_state=RANDOM_STATE
    )
    X_tr_filtered, X_holdout_filtered, y_tr_filtered, y_holdout_filtered = train_test_split(
        X_temp, y_temp, test_size=HOLDOUT_SIZE/(1-HOLDOUT_SIZE), stratify=y_temp, random_state=RANDOM_STATE
    )
    X_tr_filtered = X_tr_filtered.reset_index(drop=True)
    X_te_filtered = X_te_filtered.reset_index(drop=True)
    X_holdout_filtered = X_holdout_filtered.reset_index(drop=True)
    print(f"[{task_name}] Filtered train size: {len(y_tr_filtered)}, Test size: {len(y_te_filtered)}, Holdout size: {len(y_holdout_filtered)}")
    X_tr_full = X_df_full[used_cats + used_nums]
    y_tr_full = y_full
    print(f"[{task_name}] Full train size: {len(y_tr_full)}")
    pre = build_preprocessor(used_cats, used_nums, categories)
    X_tr_filtered_mat = pre.fit_transform(X_tr_filtered)
    colnames = list(pre.get_feature_names_out())
    best_params = {
        'n_estimators': 1500 if outcome_col == "Platelet_Engraftment" else 1200,
        'max_depth': 4,
        'learning_rate': 0.03 if outcome_col == "Platelet_Engraftment" else 0.07,
        'subsample': 0.85,
        'colsample_bytree': 0.8,
        'min_child_weight': 2,
        'reg_lambda': 1.0,
        'reg_alpha': 0.0,
        'scale_pos_weight': 1.25 if outcome_col == "Platelet_Engraftment" else 1.5
    }
    cv_summary = cv_with_threshold(X_tr_filtered, y_tr_filtered, task_name, outcome_col, used_cats, used_nums, categories, best_params)
    print(f"[{task_name}] Using fixed XGB params: {best_params}")
    
    # Leak-free preprocessing: fit only on train
    pre_final = build_preprocessor(used_cats, used_nums, categories)
    X_train_mat = pre_final.fit_transform(X_tr_filtered)
    X_val_mat   = pre_final.transform(X_holdout_filtered)
    X_test_mat  = pre_final.transform(X_te_filtered)
    colnames = list(pre_final.get_feature_names_out())

    # SMOTE only on training split
    sm = SMOTE(random_state=RANDOM_STATE)
    try:
        X_train_res, y_train_res = sm.fit_resample(X_train_mat, y_tr_filtered)
        print(f"[{task_name}] SMOTE applied: train {X_train_res.shape}")
    except ValueError as e:
        print(f"[{task_name}] SMOTE skipped: {e}")
        X_train_res, y_train_res = X_train_mat, y_tr_filtered

    # SHAP-based feature selection (training only)
    xgb_temp = safe_fit_xgb(X_train_res, y_train_res, params=best_params)
    top_features, top_indices = shap_feature_selection(xgb_temp, X_train_res, colnames)
    X_train_res = X_train_res[:, top_indices]
    X_val_mat = X_val_mat[:, top_indices]
    X_test_mat = X_test_mat[:, top_indices]

    # Train on training split only
    model = safe_fit_xgb(X_train_res, y_train_res, params=best_params)

    # Threshold tuning on validation split
    proba_val = model.predict_proba(X_val_mat)[:, 1]
    thr, _ = best_threshold_for_accuracy(y_holdout_filtered, proba_val)

    # Final evaluation on test split 
    proba_test = model.predict_proba(X_test_mat)[:, 1]
    metrics = evaluate(y_te_filtered, proba_test, prefix="test_")
    print(f"[{task_name}] TEST AUC={metrics['test_auc']:.4f} | ACC={metrics['test_acc']:.4f} | thr={metrics['test_thr']:.3f}")

    # Use the single trained model. Get probabilities for each split.
    proba_test = model.predict_proba(X_test_mat)[:, 1]
    proba_val  = model.predict_proba(X_val_mat)[:, 1]

    # Also evaluate on the "full" set using the same (train-fit) preprocessor and top_indices
    X_full_mat = pre_final.transform(X_tr_full[used_cats + used_nums])  # same features as training
    X_full_mat = X_full_mat[:, top_indices]
    proba_full = model.predict_proba(X_full_mat)[:, 1]

    # Compute metrics with consistent prefixes (filtered=test, holdout=val, full=full)
    eval_metrics_filtered = evaluate(y_te_filtered, proba_test, prefix="filtered_")
    eval_metrics_holdout  = evaluate(y_holdout_filtered, proba_val,  prefix="holdout_")
    eval_metrics_full     = evaluate(y_tr_full,        proba_full, prefix="full_")

    eval_metrics = {}
    eval_metrics.update(eval_metrics_filtered)
    eval_metrics.update(eval_metrics_holdout)
    eval_metrics.update(eval_metrics_full)
    eval_metrics.update(cv_summary)

    print(f"[{task_name}] Filtered Eval AUC={eval_metrics['filtered_auc']:.4f} | ACC={eval_metrics['filtered_acc']:.4f} | AP={eval_metrics['filtered_pr_auc']:.4f} | Brier={eval_metrics['filtered_brier']:.4f} | thr={eval_metrics['filtered_thr']:.3f} | n={len(y_te_filtered)} | prev={prevalence(y_te_filtered):.3f}")
    print(f"[{task_name}] Filtered Precision={eval_metrics['filtered_prec']:.3f} | Recall={eval_metrics['filtered_rec']:.3f} | F1={eval_metrics['filtered_f1']:.3f}")
    print(f"[{task_name}] Holdout Eval AUC={eval_metrics['holdout_auc']:.4f} | ACC={eval_metrics['holdout_acc']:.4f} | AP={eval_metrics['holdout_pr_auc']:.4f} | Brier={eval_metrics['holdout_brier']:.4f} | thr={eval_metrics['holdout_thr']:.3f} | n={len(y_holdout_filtered)} | prev={prevalence(y_holdout_filtered):.3f}")
    print(f"[{task_name}] Holdout Precision={eval_metrics['holdout_prec']:.3f} | Recall={eval_metrics['holdout_rec']:.3f} | F1={eval_metrics['holdout_f1']:.3f}")
    print(f"[{task_name}] Full Eval AUC={eval_metrics['full_auc']:.4f} | ACC={eval_metrics['full_acc']:.4f} | AP={eval_metrics['full_pr_auc']:.4f} | Brier={eval_metrics['full_brier']:.4f} | thr={eval_metrics['full_thr']:.3f} | n={len(y_tr_full)} | prev={prevalence(y_tr_full):.3f}")
    print(f"[{task_name}] Full Precision={eval_metrics['full_prec']:.3f} | Recall={eval_metrics['full_rec']:.3f} | F1={eval_metrics['full_f1']:.3f}")

    sub_df_filtered = pd.concat([
        subgroup_block(X_te_filtered, y_te_filtered, proba_test,  eval_metrics['filtered_thr'], 'Ethnicity', task_name),
        subgroup_block(X_te_filtered, y_te_filtered, proba_test,  eval_metrics['filtered_thr'], 'Race', task_name)
    ], ignore_index=True)

    sub_df_holdout = pd.concat([
        subgroup_block(X_holdout_filtered, y_holdout_filtered, proba_val, eval_metrics['holdout_thr'], 'Ethnicity', task_name),
        subgroup_block(X_holdout_filtered, y_holdout_filtered, proba_val, eval_metrics['holdout_thr'], 'Race', task_name)
    ], ignore_index=True)

    sub_df_full = pd.concat([
        subgroup_block(X_df_full, y_full, proba_full, eval_metrics['full_thr'], 'Ethnicity', task_name),
        subgroup_block(X_df_full, y_full, proba_full, eval_metrics['full_thr'], 'Race', task_name)
    ], ignore_index=True)

    if not sub_df_filtered.empty:
        print(f"[{task_name}] Filtered subgroup snapshot:")
        print(sub_df_filtered.sort_values(['group', 'n'], ascending=[True, False]))
    if not sub_df_holdout.empty:
        print(f"[{task_name}] Holdout subgroup snapshot:")
        print(sub_df_holdout.sort_values(['group', 'n'], ascending=[True, False]))
    if not sub_df_full.empty:
        print(f"[{task_name}] Full subgroup snapshot:")
        print(sub_df_full.sort_values(['group', 'n'], ascending=[True, False]))
    for gcol in ["Ethnicity", "Race"]:
        if gcol in X_te_filtered.columns:
            plot_acc_prev_for_group(X_te_filtered, y_te_filtered, proba_test, gcol, eval_metrics['filtered_thr'], f"{task_name}_filtered", FIG_DIR)
        if gcol in X_holdout_filtered.columns:
            plot_acc_prev_for_group(X_holdout_filtered, y_holdout_filtered, proba_val, gcol, eval_metrics['holdout_thr'], f"{task_name}_holdout", FIG_DIR)
        if gcol in X_df_full.columns:
            plot_acc_prev_for_group(X_df_full, y_full, proba_full, gcol, eval_metrics['full_thr'], f"{task_name}_full", FIG_DIR)

    plot_roc(model, X_test_mat,  y_te_filtered,      f"ROC: {task_name} (Filtered)", out_png(f"roc_{task_name}_filtered"))
    print(f"[{task_name}] Saved ROC plot (filtered) to: {OUT_DIR / f'roc_{task_name}_filtered.png'}")

    plot_roc(model, X_val_mat,   y_holdout_filtered, f"ROC: {task_name} (Holdout)",  out_png(f"roc_{task_name}_holdout"))
    print(f"[{task_name}] Saved ROC plot (holdout) to: {OUT_DIR / f'roc_{task_name}_holdout.png'}")

    plot_roc(model, X_full_mat,  y_tr_full,          f"ROC: {task_name} (Full)",     out_png(f"roc_{task_name}_full"))
    print(f"[{task_name}] Saved ROC plot (full) to: {OUT_DIR / f'roc_{task_name}_full.png'}")

    try:
        explainer = shap.TreeExplainer(model)
        shap_values = explainer.shap_values(X_test_mat)
        shap.summary_plot(shap_values, X_test_mat, feature_names=top_features, show=False)
        plt.savefig(OUT_DIR / f"shap_summary_{task_name}_filtered", dpi=220, bbox_inches="tight")
        plt.close()
        print(f"[{task_name}] Saved SHAP plot (filtered) to: {OUT_DIR / f'shap_summary_{task_name}_filtered.png'}")
    except Exception as e:
        print(f"[{task_name}] SHAP plot (filtered) failed: {e}")

    try:
        shap_values = explainer.shap_values(X_val_mat)
        shap.summary_plot(shap_values, X_val_mat, feature_names=top_features, show=False)
        plt.savefig(OUT_DIR / f"shap_summary_{task_name}_holdout", dpi=220, bbox_inches="tight")
        plt.close()
        print(f"[{task_name}] Saved SHAP plot (holdout) to: {OUT_DIR / f'shap_summary_{task_name}_holdout.png'}")
    except Exception as e:
        print(f"[{task_name}] SHAP plot (holdout) failed: {e}")

    try:
        shap_values = explainer.shap_values(X_full_mat)
        shap.summary_plot(shap_values, X_full_mat, feature_names=top_features, show=False)
        plt.savefig(OUT_DIR / f"shap_summary_{task_name}_full", dpi=220, bbox_inches="tight")
        plt.close()
        print(f"[{task_name}] Saved SHAP plot (full) to: {OUT_DIR / f'shap_summary_{task_name}_full.png'}")
    except Exception as e:
        print(f"[{task_name}] SHAP plot (full) failed: {e}")

    metrics_path = out_csv(f"metrics_{task_name}")
    pd.DataFrame([eval_metrics]).assign(model="XGBoost", task=task_name).to_csv(metrics_path, index=False)
    print(f"[{task_name}] Saved metrics to: {metrics_path}")

    joblib.dump(model, out_pkl(f"ucbt_xgb_model_{task_name}"))
    with open(out_json(f"threshold_{task_name}"), "w") as f:
        json.dump({"threshold": float(eval_metrics['filtered_thr'])}, f)

    print(f"[{task_name}] Saved threshold to: {OUT_DIR / f'{task_name}_threshold.json'}")
    if not sub_df_filtered.empty:
        p = out_csv(f"subgroup_{task_name}_filtered"); sub_df_filtered.to_csv(p, index=False); print(f"[{task_name}] Saved filtered subgroup to: {p}")
        print(f"[{task_name}] Saved filtered subgroup results to: {OUT_DIR / f'{task_name}_subgroup_filtered.csv'}")
    if not sub_df_holdout.empty:
        p = out_csv(f"subgroup_{task_name}_holdout");  sub_df_holdout.to_csv(p, index=False);  print(f"[{task_name}] Saved holdout subgroup to: {p}")
        print(f"[{task_name}] Saved holdout subgroup results to: {OUT_DIR / f'{task_name}_subgroup_holdout.csv'}")
    if not sub_df_full.empty:
        p = out_csv(f"subgroup_{task_name}_full");     sub_df_full.to_csv(p, index=False);     print(f"[{task_name}] Saved full subgroup to: {p}")
        print(f"[{task_name}] Saved full subgroup results to: {OUT_DIR / f'{task_name}_subgroup_full.csv'}")
    return {
        'task': task_name,
        'outcome': outcome_col,
        'n_train': len(y_tr_full),
        'n_test_filtered': len(y_te_filtered),
        'n_holdout_filtered': len(y_holdout_filtered),
        'n_test_full': len(y_full),
        'filtered_auc': eval_metrics['filtered_auc'],
        'filtered_acc': eval_metrics['filtered_acc'],
        'filtered_thr': eval_metrics['filtered_thr'],
        'filtered_prev': prevalence(y_te_filtered),
        'filtered_ap': eval_metrics['filtered_pr_auc'],
        'filtered_precision': eval_metrics['filtered_prec'],
        'filtered_recall': eval_metrics['filtered_rec'],
        'filtered_f1': eval_metrics['filtered_f1'],
        'holdout_auc': eval_metrics['holdout_auc'],
        'holdout_acc': eval_metrics['holdout_acc'],
        'holdout_thr': eval_metrics['holdout_thr'],
        'holdout_prev': prevalence(y_holdout_filtered),
        'holdout_ap': eval_metrics['holdout_pr_auc'],
        'holdout_precision': eval_metrics['holdout_prec'],
        'holdout_recall': eval_metrics['holdout_rec'],
        'holdout_f1': eval_metrics['holdout_f1'],
        'full_auc': eval_metrics['full_auc'],
        'full_acc': eval_metrics['full_acc'],
        'full_thr': eval_metrics['full_thr'],
        'full_prev': prevalence(y_full),
        'full_ap': eval_metrics['full_pr_auc'],
        'full_precision': eval_metrics['full_prec'],
        'full_recall': eval_metrics['full_rec'],
        'full_f1': eval_metrics['full_f1'],
        'cv_auc_mean': cv_summary['auc_mean'],
        'cv_acc_mean': cv_summary['acc_mean'],
        'cv_auc_std': cv_summary['auc_std'],
        'cv_acc_std': cv_summary['acc_std']
    }

# MASKS
def dose_filter_mask(d):
    ok = pd.Series(True, index=d.index)
    if "CD34_num" in d.columns:
        s = pd.to_numeric(d["CD34_num"], errors="coerce")
        ok &= (s >= CD34_MIN) & (s <= CD34_MAX_PLATELET) & s.notna()
        print(f"[dose_filter_mask] CD34_num: min={s.min():.2f}, max={s.max():.2f}, nulls={s.isna().sum()}")
    if "TNC_num" in d.columns:
        s = pd.to_numeric(d["TNC_num"], errors="coerce")
        ok &= (s >= TNC_MIN) & (s <= TNC_MAX_PLATELET) & s.notna()
        print(f"[dose_filter_mask] TNC_num: min={s.min():.2f}, max={s.max():.2f}, nulls={s.isna().sum()}")
    if "Conditioning_Regimen" in d.columns:
        ok &= (d["Conditioning_Regimen"].isin(['MA', 'RIC'])) & d["Conditioning_Regimen"].notna()
        print(f"[dose_filter_mask] Conditioning_Regimen counts: {d['Conditioning_Regimen'].value_counts().to_dict()}")
    return ok
    
def all_only_mask(d):
    ok = (d["Disease_Type"] == "ALL") & d["Disease_Type"].notna() & (d["Disease_Type"] != "Unknown")
    if "CD34_num" in d.columns:
        s = pd.to_numeric(d["CD34_num"], errors="coerce")
        ok &= (s >= CD34_MIN) & (s <= CD34_MAX_SURVIVAL) & s.notna()
        print(f"[all_only_mask] CD34_num: min={s.min():.2f}, max={s.max():.2f}, nulls={s.isna().sum()}")
    if "TNC_num" in d.columns:
        s = pd.to_numeric(d["TNC_num"], errors="coerce")
        ok &= (s >= TNC_MIN) & (s <= TNC_MAX_SURVIVAL) & s.notna()
        print(f"[all_only_mask] TNC_num: min={s.min():.2f}, max={s.max():.2f}, nulls={s.isna().sum()}")
    if "Conditioning_Regimen" in d.columns:
        ok &= (d["Conditioning_Regimen"].isin(['MA', 'RIC'])) & d["Conditioning_Regimen"].notna()
        print(f"[all_only_mask] Conditioning_Regimen counts: {d['Conditioning_Regimen'].value_counts().to_dict()}")
    if "HLA_Match_Level" in d.columns:
        ok &= (d["HLA_Match_Level"].isin(['4/6', '5/6', '6/6'])) & d["HLA_Match_Level"].notna()
        print(f"[all_only_mask] HLA_Match_Level counts: {d['HLA_Match_Level'].value_counts().to_dict()}")
    print(f"[all_only_mask] Disease_Type counts: {d['Disease_Type'].value_counts().to_dict()}")
    return ok
    
def default_train_mask_platelet(d):
    return d["Platelet_Engraftment"].notna() & dose_filter_mask(d) if USE_DOSE_FILTERS else d["Platelet_Engraftment"].notna()

def eval_mask_platelet(d):
    return d["Platelet_Engraftment"].notna() & dose_filter_mask(d) if USE_DOSE_FILTERS else d["Platelet_Engraftment"].notna()

def default_train_mask_survival(d):
    return d["1_Year_Survival"].notna() & all_only_mask(d) if USE_ALL_ONLY_SURVIVAL else d["1_Year_Survival"].notna()

def eval_mask_survival(d):
    return d["1_Year_Survival"].notna() & all_only_mask(d) if USE_ALL_ONLY_SURVIVAL else d["1_Year_Survival"].notna()

# FEATURE SETS
CAT_COLS = ["Disease_Type", "Conditioning_Regimen", "HLA_Match_Level", "Ethnicity", "Race"]
NUM_COLS = [
    "Recipient_Age", "CD34_num", "TNC_num", "HLAxCD34", "Age_x_CD34", "Age_x_TNC", "Regimen_MA", "Remission_Status"
]

# LOAD DATA
print(f"Loading: {INPUT_PATH}")
df = pd.read_csv(INPUT_PATH)
print("Loaded:", df.shape)

# Validate synthetic data
if os.path.exists(REAL_PATH):
    df_real = pd.read_csv(REAL_PATH)
    validate_synthetic_data(df, df_real)
else:
    print(f"Warning: Real dataset not found at {REAL_PATH}. Skipping KS validation.")
    
# Preprocess data
category_mappings = {
    "HLA_Match_Level": {
        '4/6,4/6': '4/6', '4/6,5/6': '4/6', '4/6,6/6': '4/6',
        '5/6,5/6': '5/6', '5/6,6/6': '5/6', '6/6,6/6': '6/6',
        '<=4/6': '4/6', '>=4/6': '4/6', '>=5/6': '5/6', '<=5/8': '5/6',
        '4/6 or 5/6': '4/6', '6-8/8': '6/6', 'nan': 'Unknown', '': 'Unknown',
        'Missing/Unknown': 'Unknown', 'Missing': 'Unknown'
    },
    "Disease_Type": {
        '': 'Unknown', 'nan': 'Unknown'
    },
    "Conditioning_Regimen": {
        '': 'Unknown', 'nan': 'Unknown', 'Myeloablative': 'MA',
        'Non-myeloablative': 'RIC', 'Reduced intensity': 'RIC', 'Reduced Intensity': 'RIC'
    },
    "Ethnicity": {
        '': 'Unknown', 'nan': 'Unknown', 'Other/Unknown': 'Unknown',
        'Hispanic': 'Hispanic/Latinx', 'Not Hispanic or Latino': 'Non-Hispanic',
        'Hispanic or Latino': 'Hispanic/Latinx', 'Japanese': 'Asian',
        'Other': 'Unknown', 'Unknown/Missing': 'Unknown'
    },
    "Race": {
        '': 'Unknown', 'nan': 'Unknown', 'Missing/Unknown': 'Unknown',
        'Caucasian(white)': 'Caucasian', 'African-American(Black)': 'African American',
        'Non-White': 'Other', 'Non-Caucasian': 'Other', 'Native American': 'Other',
        'Black or African American': 'African American', 'White': 'Caucasian',
        'Black': 'African American', 'Unknown/Missing': 'Unknown',
        'More than one race': 'Other', 'American Indian or Alaskan Native': 'Other',
        'Others': 'Other'
    }
}
for col, mapping in category_mappings.items():
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip().replace(mapping)
        print(f"Unique {col} after mapping: {df[col].unique().tolist()}")
categories = []
for col in CAT_COLS:
    if col in df.columns:
        unique_vals = sorted(df[col].unique().tolist())
        categories.append(unique_vals)
        print(f"Predefined categories for {col}: {unique_vals}")
print("Verifying filter application...")
dose_mask = dose_filter_mask(df)
all_mask = all_only_mask(df)

print(f"Platelet_Engraftment rows after dose filter: {df[dose_filter_mask(df) & df['Platelet_Engraftment'].notna()].shape[0]}")
print(f"1_Year_Survival rows after ALL-only filter: {df[all_only_mask(df) & df['1_Year_Survival'].notna()].shape[0]}")

# TASK LIST
tasks = [
    (
        "Platelet_Engraftment_FullDose",
        "Platelet_Engraftment",
        default_train_mask_platelet,
        eval_mask_platelet,
        CAT_COLS, NUM_COLS, [], categories
    ),
    (
        "1_Year_Survival_FullALL",
        "1_Year_Survival",
        default_train_mask_survival,
        eval_mask_survival,
        CAT_COLS, NUM_COLS, [], categories
    )
]

# RUN
rows = []
for name, outcome, tr_mask, ev_mask, cat_cols, num_cols, excl, cats in tasks:
    res = run_task(name, outcome, tr_mask, ev_mask, cat_cols, num_cols, excl, cats)
    rows.append(res)
summary = pd.DataFrame(rows).sort_values(["filtered_auc", "filtered_acc"], ascending=[False, False])
print("\nSummary:")
print(summary[["task", "outcome", "n_train", "n_test_filtered", "n_holdout_filtered", "filtered_auc", "filtered_acc", "holdout_auc", "holdout_acc", "filtered_thr", "filtered_prev", "filtered_ap", "full_auc", "full_acc", "cv_auc_mean", "cv_acc_mean"]])
summary_path = out_csv("summary")
summary.to_csv(summary_path, index=False)
print(f"[saved] {summary_path}")


Python executable: /opt/anaconda3/bin/python
SHAP version: 0.46.0
Loading: /Users/amanda/Desktop/UCBT/ucbt_dataset_synthetic_best.csv
Loaded: (20000, 63)

Validating synthetic data with Kolmogorov-Smirnov tests...
KS test for CD34_num: stat=0.0532, p=0.0000
KS test for TNC_num: stat=0.0528, p=0.0000
KS test for Recipient_Age: stat=0.0115, p=0.3324
KS test for HLA_Match_Level distribution: stat=0.1111, p=1.0000
KS test for Disease_Type distribution: stat=0.2000, p=1.0000
KS test for Conditioning_Regimen distribution: stat=0.2500, p=1.0000
KS test for Ethnicity distribution: stat=0.2222, p=0.9895
KS test for Race distribution: stat=0.3333, p=0.2754
[saved] /Users/amanda/Desktop/UCBT/models_output/metrics/ks_validation_results_synthetic_20251027_165802.csv
Unique HLA_Match_Level after mapping: ['5/6', '4/6', '6/6', '3/6', 'Unknown', '2/6']
Unique Disease_Type after mapping: ['AML', 'MDS', 'ALL', 'Other', 'CML']
Unique Conditioning_Regimen after mapping: ['MA', 'RIC', 'Other', 'Unknown']
U